In [0]:
%run ./_config

In [0]:
# metadata table to track api extraction progress
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE_SCHEMA}.ingestion_metadata (
        source_name STRING,
        last_loaded_date DATE,
        rows_loaded BIGINT,
        load_timestamp TIMESTAMP
    )
""")
print(f"Metadata table ready - {BRONZE_SCHEMA}.ingestion_metadata")

In [0]:
# get high watermark/last loaded date for incremental api calls
watermark_df = spark.sql(f"""
    SELECT MAX(last_loaded_date) AS max_date
    FROM {BRONZE_SCHEMA}.ingestion_metadata
    WHERE source_name = '{RECORD_SOURCE}'
""")

last_date = watermark_df.collect()[0]["max_date"]

if last_date is None:
    start_date = "2025-01-01"
    print(f"1st run, loading from {start_date}")
else:
    start_date = str(last_date)
    print(f"Incremental run, loading from {start_date}")

end_date = "2025-12-31"

In [0]:
# fetch data from api, write .json files to volume
import urllib.request
import urllib.parse
import json
import time
from datetime import datetime

BASE_URL = "https://data.iowa.gov/resource/m3tr-qhgy.json"
PAGE_SIZE = 50000

batch_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"{LANDING_ZONE}/liquor_sales/{batch_ts}"
dbutils.fs.mkdirs(output_dir)

where_clause = f"date >= '{start_date}T00:00:00' AND date <= '{end_date}T23:59:59'"

offset = 0
total_rows = 0
page_num = 0
max_date_seen = start_date

print(f"Starting ingestion: {start_date} to {end_date}")
print(f"Output directory: {output_dir}")
print(f"Page size: {PAGE_SIZE:,}")
print("-" * 60)

while True:
    params = urllib.parse.urlencode(
        {
            "$where": where_clause,
            "$limit": PAGE_SIZE,
            "$offset": offset,
            "$order": "date ASC, invoice_line_no ASC", 
        },
        quote_via=urllib.parse.quote
    )
    url = f"{BASE_URL}?{params}"

    req = urllib.request.Request(url)
    req.add_header("Accept", "application/json")

    try:
        with urllib.request.urlopen(req, timeout=60) as response:
            data = json.loads(response.read().decode())
    except urllib.error.HTTPError as e:
        body = e.read().decode()
        print(f"Problem on page {page_num}: {e} — {body}")
        print("Retry in 5 seconds...")
        time.sleep(5)
        try:
            with urllib.request.urlopen(req, timeout=60) as response:
                data = json.loads(response.read().decode())
        except Exception as e2:
            print(f"Retry failed: {e2}")
            break
    except Exception as e:
        print(f"Error on page {page_num}: {e}")
        break

    rows_returned = len(data)
    if rows_returned == 0:
        print("No more rows")
        break

    for row in data:
        if "date" in row and row["date"]:
            row_date = row["date"][:10]
            if row_date > max_date_seen:
                max_date_seen = row_date

    file_path = f"{output_dir}/page_{page_num:04d}.json"
    dbutils.fs.put(file_path, json.dumps(data), overwrite=True)

    total_rows += rows_returned
    page_num += 1
    print(f"  Page {page_num}: {rows_returned:,} rows: {file_path}")

    if rows_returned < PAGE_SIZE:
        print(f"Last page (got {rows_returned} < {PAGE_SIZE}). Done.")
        break

    offset += PAGE_SIZE
    time.sleep(0.5)

print("-" * 60)
print(f"Total rows loaded: {total_rows:,}")
print(f"Total pages: {page_num}")
print(f"Latest date: {max_date_seen}")

In [0]:
# record load in the metadata table
from datetime import datetime

if total_rows > 0:
    spark.sql(f"""
        INSERT INTO {BRONZE_SCHEMA}.ingestion_metadata
        VALUES (
            '{RECORD_SOURCE}',
            DATE '{max_date_seen}',
            {total_rows},
            current_timestamp()
        )
    """)
    print(f"Metadata updated: {total_rows:,} rows, max date = {max_date_seen}")

    # set values for downstream quality checks
    try:
        dbutils.jobs.taskValues.set(key="rows_ingested", value=total_rows)
        dbutils.jobs.taskValues.set(key="load_date", value=max_date_seen)
    except Exception:
        pass
else:
    print("No new data loaded")

In [0]:
# list files in the landing zone, sanity check
files = dbutils.fs.ls(f"{LANDING_ZONE}/liquor_sales/{batch_ts}/")
print(f"Files in landing zone ({len(files)}):")
for f in files[:10]:
    print(f"  {f.name} — {f.size:,} bytes")
if len(files) > 10:
    print(f"  ... and {len(files) - 10} more")

In [0]:
# column names, sanity check
# print(list(data[0].keys()))